# exploracao_medicao

Exploração da tabela `bronze.tb_leituras` para levantamento de problemas de qualidade e definição das regras do `silver_medicao`.
- **Seção 1:** Perfilamento da **Bronze bruta (ANTES de qualquer tratamento)** — dados ainda como `string`.
- **Seção 2:** Conversão de Tipos (`parse_data`, `parse_decimal`, `valores_para_nulo`) — transforma texto em `date` e `decimal` para inspecionar estatísticas e identificar anomalias como `consumo_kwh < 0`.
- **Seção 3:** Decomposição das **3 causas dos consumos negativos** (incluindo os 6.215 casos de 1ª leitura pós-troca de medidor), domínios, duplicatas e órfãos.

In [0]:
%run ./funcoes_comuns

In [0]:
CATALOGO = "workspace"

lt = ler_bronze(CATALOGO, "leituras")
print(f"{lt.count():,} linhas na Bronze (dados brutos sem tratamento)")
lt.printSchema()

COLUNAS_LT = [c for c in lt.columns if not c.startswith("_")]
print("Colunas de negócio:", COLUNAS_LT)

## 1. Raio-X da Bronze Bruta (ANTES do tratamento)
> **Nota:** Nas células abaixo, os dados estão em estado bruto (`string` na Bronze). Formatos de data mistos como `28/12/2025` (`dd/MM/yyyy`) ou `01-03-2024` (`MM-dd-yyyy`) e valores como `-1` ou `N/A` aparecem aqui justamente para comprovar a sujeira da origem.

In [0]:
# [ANTES DO TRATAMENTO] Perfilamento das colunas brutas (ainda como string na Bronze)
print(">>> PERFILAMENTO BRUTO (BRONZE - ANTES DO TRATAMENTO DE TIPOS E DATAS) <<<")
for c in COLUNAS_LT:
    perfilar_coluna(lt, c, n=10)

In [0]:
# [ANTES DO TRATAMENTO] Checagem de estrutura: cabeçalho repetido e espaços nas pontas
from functools import reduce

n_total = lt.count()
lt_sem_cab = remover_cabecalho_repetido(lt)
print(f"Cabeçalhos repetidos no meio dos arquivos: {n_total - lt_sem_cab.count():,}")

tem_espaco = reduce(lambda a, b: a | b, [F.col(c).rlike(r"^\s|\s$") for c in COLUNAS_LT])
print(f"Linhas com espaços nas pontas: {lt_sem_cab.filter(tem_espaco).count():,}")

por_coluna_espaco = lt_sem_cab.agg(*[
    F.sum(F.col(c).rlike(r"^\s|\s$").cast("int")).alias(c) for c in COLUNAS_LT
]).first().asDict()
print("Espaços nas pontas por coluna:", {k: v for k, v in por_coluna_espaco.items() if v})

In [0]:
# [ANTES DO TRATAMENTO] Contagem das formas de 'sem valor' em leitura_anterior e formatos de data na Bronze
lt_aparado = aparar_textos(lt_sem_cab)

if "leitura_anterior" in COLUNAS_LT:
    nao_numerico_ant = lt_aparado.filter(
        F.col("leitura_anterior").isNull()
        | ~F.col("leitura_anterior").rlike(r"^\d+(\.\d+|,\d+)?$")
    )
    print(f"[BRONZE BRUTA] leitura_anterior nula ou com marcador de ausência ('N/A', 'NULL', '-1'): {nao_numerico_ant.count():,} linhas")
    display(nao_numerico_ant.groupBy("leitura_anterior").count().orderBy(F.desc("count")))

col_data = next((c for c in ["data_leitura", "data_medicao"] if c in COLUNAS_LT), None)
if col_data:
    fora_iso = lt_aparado.filter(F.col(col_data).isNotNull() & ~F.col(col_data).rlike(r"^\d{4}-\d{2}-\d{2}$"))
    print(f"[BRONZE BRUTA] {col_data} fora do formato ISO yyyy-MM-dd (ex: dd/MM/yyyy, MM-dd-yyyy, dd.MM.yyyy): {fora_iso.count():,} linhas")

## 2. Conversão de Tipos (`valores_para_nulo`, `parse_data`, `parse_decimal`)
> **Nota:** Aqui convertemos o texto da Bronze em `date` e `decimal(14,2)` (igual à Célula 8 do `exploracao_cadastro`). Isso **ainda é antes de corrigir as anomalias de negócio**: convertemos o tipo justamente para o `perfilar_coluna` calcular `min`, `max` e `summary` e revelar os consumos negativos (que serão decompostos nas 3 causas e recuperados na Seção 3).

In [0]:
# [APÓS CONVERSÃO DE TIPOS, ANTES DE CORRIGIR CONSUMO NEGATIVO] Inspeciona estatísticas numéricas e datas
VALORES_NULOS_LEITURA = VALORES_NULOS + ("-1",)

cols_decimais = [c for c in ["leitura_anterior", "leitura_atual", "consumo_kwh", "consumo", "consumo_apurado"] if c in COLUNAS_LT]
cols_datas = [c for c in ["data_leitura", "data_medicao"] if c in COLUNAS_LT]

lt_limpo_nulos = lt_aparado
for c in cols_decimais:
    lt_limpo_nulos = lt_limpo_nulos.withColumn(c, valores_para_nulo(c, VALORES_NULOS_LEITURA if c == "leitura_anterior" else VALORES_NULOS))

if cols_decimais:
    checar_parse(lt_limpo_nulos, cols_decimais, parse_decimal, "leituras_pos_parse")
if cols_datas:
    checar_parse(lt_limpo_nulos, cols_datas, parse_data, "leituras_pos_parse")

lt_tipos = lt_limpo_nulos
for c in cols_decimais:
    lt_tipos = lt_tipos.withColumn(c, parse_decimal(c))
for c in cols_datas:
    lt_tipos = lt_tipos.withColumn(c, parse_data(c))

print("\n>>> PERFILAMENTO APÓS CONVERSÃO DE TIPOS (ANTES DE RECUPERAR OS CONSUMOS NEGATIVOS) <<<")
for c in cols_decimais + cols_datas:
    perfilar_coluna(lt_tipos, c, n=5)

## 3. Investigação de Domínios, Duplicatas, Órfãos e as 3 Causas dos Consumos Negativos

In [0]:
# 3.1 Domínios (tipo_leitura, mes_referencia / competencia) antes vs depois de padronizar
for c in ["tipo_leitura", "mes_referencia", "competencia", "status_leitura", "ocorrencia_leitura"]:
    if c in COLUNAS_LT:
        print(f"=== Domínio de {c} (Bruto vs Padronizado) ===")
        display(lt_tipos.groupBy(c, padronizar_texto(c).alias(f"{c}_padronizado")).count().orderBy(F.desc("count")))

In [0]:
# 3.2 Deduplicação e separação de órfãos contra silver.tb_medidores
lt_pad = lt_tipos
for c in [c for c in COLUNAS_LT if c not in cols_decimais + cols_datas]:
    lt_pad = lt_pad.withColumn(c, padronizar_texto(c))

n_antes_dedup = lt_pad.count()
lt_dedup = deduplicar(lt_pad)
n_depois_dedup = lt_dedup.count()
print(f"Duplicatas idênticas nas colunas de negócio: {n_antes_dedup - n_depois_dedup:,} (restam {n_depois_dedup:,})")

md_silver = spark.table(f"{CATALOGO}.silver.tb_medidores")
lt_validos, lt_orfaos = separar_orfaos(lt_dedup, md_silver, "id_medidor")
print(f"Por id_medidor -> Válidos: {lt_validos.count():,} | Órfãos: {lt_orfaos.count():,}")

In [0]:
# 3.3 Decomposição das 3 causas de consumo_kwh < 0 nos dados deduplicados e sem órfãos
md_ref = md_silver.select("id_medidor", "data_instalacao", "data_retirada", "tipo", "status")
lt_com_md = lt_validos.join(md_ref, "id_medidor", "left")

dias_apos_instalacao = F.datediff(F.col("data_leitura"), F.col("data_instalacao"))
pos_troca = (
    (F.col("consumo_kwh") < 0)
    & (F.col("data_instalacao") >= F.lit("2025-09-01"))
    & (dias_apos_instalacao.between(0, 45))
)

neg_df = lt_com_md.filter(F.col("consumo_kwh") < 0).withColumn(
    "dias_apos_instalacao", dias_apos_instalacao
).withColumn(
    "causa_hipotese",
    F.when(pos_troca, F.lit("1_POS_TROCA_MEDIDOR (ate 45d)"))
     .when(F.col("leitura_atual") >= F.col("leitura_anterior"), F.lit("2_SINAL_INVERTIDO (atual >= anterior)"))
     .when(F.col("consumo_kwh") >= -5, F.lit("3A_PEQUENO_AJUSTE_ARREDONDAMENTO (>= -5 kWh)"))
     .otherwise(F.lit("3B_OUTROS_ATUAL_MENOR_ANTERIOR"))
)

print(f"Total de leituras válidas com consumo_kwh < 0: {neg_df.count():,}")
display(
    neg_df.groupBy("causa_hipotese", "tipo_leitura")
    .agg(
        F.count("*").alias("qtd_linhas"),
        F.min("consumo_kwh").alias("min_consumo"),
        F.max("consumo_kwh").alias("max_consumo"),
        F.sum("leitura_atual").alias("soma_leitura_atual_kwh")
    )
    .orderBy(F.desc("qtd_linhas"))
)

print("Exemplos dos que NÃO são pós-troca de medidor:")
display(neg_df.filter(~pos_troca).orderBy("consumo_kwh").limit(30))

## Resultados da exploração (Medição - `leituras`)

### Leituras (`bronze.tb_leituras`)

| Achado na Bronze (Antes do Tratamento) | Quantidade | Regra na Silver (`silver_medicao`) |
|---|---|---|
| Cabeçalho repetido no meio dos arquivos CSV | *(preencher após rodar)* | `remover_cabecalho_repetido` (`REMOVIDO`) |
| Espaços no início/fim dos campos de texto | *(preencher após rodar)* | `aparar_textos` (`CORRIGIDO`) |
| `leitura_anterior` sem valor em 4 formas: vazio, `N/A`, `NULL` e `-1` | ~32.700 linhas | `valores_para_nulo("leitura_anterior", VALORES_NULOS + ("-1",))` → nulo (`CORRIGIDO`) |
| Datas de leitura em 4 formatos na Bronze (`yyyy-MM-dd`, `dd/MM/yyyy`, `dd.MM.yyyy`, `MM-dd-yyyy`) | *(preencher após rodar)* | `parse_data` converte 100% para `date` (`yyyy-MM-dd`) validado com `checar_parse` (`CORRIGIDO`) |
| Valores decimais (`leitura_anterior`, `leitura_atual`, `consumo_kwh`) em texto | *(preencher após rodar)* | `parse_decimal(14, 2)` com validação `checar_parse` (`CORRIGIDO`) |
| Grafias variadas em `tipo_leitura` (caixa/acentos) | *(preencher após rodar)* | `mapear_dominio` / `padronizar_texto` (`CORRIGIDO`) |
| Duplicatas idênticas entre/dentro dos 5 arquivos anuais (`leituras_2022` a `2026`) | *(preencher após rodar)* | `deduplicar(df)` (`REMOVIDO`) |
| 1ª leitura pós-troca de medidor com `consumo_kwh < 0` (`leitura_anterior` herdou acumulado do medidor antigo) | 6.215 linhas | `consumo_kwh = leitura_atual` (`CORRIGIDO`) + `flag_primeira_leitura_pos_troca` (`FLAG`) |
| Demais causas de `consumo_kwh < 0` | *(preencher após rodar célula 3.3)* | *(definir conforme as 3 causas)* |
| Registros órfãos em relação ao cadastro (`silver.tb_medidores` / `silver.tb_unidades_consumidoras`) | *(preencher após rodar)* | `separar_orfaos` → `silver.tb_leituras_quarentena` (`QUARENTENA`) |
